In [1]:

!pip install -q pymupdf pandas google-genai


In [2]:

import os
import json
import pandas as pd
import fitz

from google import genai
from google.colab import userdata
from IPython.display import display


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:

from google.colab import files

uploaded = files.upload()


Saving contractor_material_labour.pdf to contractor_material_labour.pdf
Saving contractor_lumpsum.pdf to contractor_lumpsum.pdf
Saving contractor_boq.pdf to contractor_boq.pdf


In [4]:

quotation_texts = {}

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        doc = fitz.open(filename)

        text = ""
        for page in doc:
            text += page.get_text() + "\n"

        quotation_texts[filename] = text
        doc.close()

        print(f"\nExtracted: {filename}")
        print(f"Characters: {len(text)}")



Extracted: contractor_material_labour.pdf
Characters: 8943

Extracted: contractor_lumpsum.pdf
Characters: 8864

Extracted: contractor_boq.pdf
Characters: 8771


In [5]:

for filename, text in quotation_texts.items():
    print("\n" + "=" * 80)
    print("FILE:", filename)
    print("=" * 80)
    print(text[:1500])



FILE: contractor_material_labour.pdf
VAZ PATIL CONSTRUCTIONS LLP
Civil Contractors | Construction & Project Management
Shop No. 6, Sai Arcade, Near Municipal Garden, Verna, Salcete, Goa – 403722
Phone: +91 83906 45218 | Email: estimates@vazpatilconstructions.in
GSTIN: 30AAFFV3392Q1ZP | PAN: AAFFV3392Q
QUOTATION FOR CIVIL CONSTRUCTION WORK
Quotation No.: VPC/2026/076
Date: 09/11/2026
Validity: 30 Days (up to 09/12/2026)
TO: CLIENT DETAILS
Client Name: Mr. Siddharth V. Kamat
Address: Flat B-402, Marigold Heights, Porvorim, Bardez, Goa – 403521
Contact: +91 98230 55126
PROJECT DETAILS
Project Name: Proposed G+1 Residential House for Mr. Siddharth V. Kamat
Project Location: Survey No. 87/3, Porvorim, Bardez, Goa – 403521
Building Type: Residential (Independent House – 3 BHK)
Built-up Area: 2,000 sq. ft. (Ground Floor 1,000 sq. ft. + First Floor 1,000 sq. ft.)
Number of Floors: G+1 (Ground + 1 Floor)
1. SCOPE OF WORK
We are pleased to submit our quotation for the execution of the above-men

In [6]:

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini API connected successfully!")


Gemini API connected successfully!


In [7]:

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: Gemini connection successful"
)

print(response.text)


Gemini connection successful


In [8]:

def extract_quotation_data(filename, quotation_text):

    prompt = f"""
You are an expert construction quotation analyst familiar
with residential construction quotations in Goa, India.

Analyze the following contractor quotation.

Extract the information into valid JSON.

Return this structure:

{{
    "contractor": "Contractor name",
    "project": "Project description",
    "total_amount": 0,
    "gst": "GST details",
    "completion_period": "Completion timeline",
    "warranty": "Warranty details",
    "payment_terms": "Payment terms",
    "items": [
        {{
            "description": "Original item description",
            "quantity": 0,
            "unit": "Unit",
            "rate": 0,
            "amount": 0
        }}
    ]
}}

Important:
- Extract only information present in the quotation.
- Do not invent missing values.
- Use null for unavailable numerical values.
- Preserve original item descriptions.
- Return JSON only, without Markdown.

QUOTATION:
{quotation_text}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    result = response.text.strip()

    if result.startswith("```"):
        result = result.split("\n", 1)[1]
        result = result.rsplit("```", 1)[0].strip()

    return json.loads(result)


In [9]:

extracted_quotations = {}

for filename, text in quotation_texts.items():

    print(f"Processing: {filename}")

    extracted_quotations[filename] = extract_quotation_data(
        filename, text
    )

    print("Successfully extracted!")


Processing: contractor_material_labour.pdf
Successfully extracted!
Processing: contractor_lumpsum.pdf
Successfully extracted!
Processing: contractor_boq.pdf
Successfully extracted!


In [10]:

for filename, data in extracted_quotations.items():

    print("\nCONTRACTOR:", data.get("contractor"))
    print("PROJECT:", data.get("project"))
    print("TOTAL AMOUNT:", data.get("total_amount"))
    print("TIMELINE:", data.get("completion_period"))
    print("WARRANTY:", data.get("warranty"))
    print("PAYMENT:", data.get("payment_terms"))

    print("\nITEMS:")
    display(pd.DataFrame(data.get("items", [])))



CONTRACTOR: Vaz Patil Constructions LLP
PROJECT: Proposed G+1 Residential House for Mr. Siddharth V. Kamat
TOTAL AMOUNT: 5172318
TIMELINE: 9 Months from the date of commencement
WARRANTY: Defects liability period of 12 months from the date of handing over for workmanship defects.
PAYMENT: Advance Payment: 25% on acceptance of quotation / issue of work order; During Execution: 65% released stage-wise; On Completion: 10% on completion and handing over of the building

ITEMS:


,description,quantity,unit,rate,amount
0,"Cement – Ambuja OPC 53 Grade, 50 kg bags (RCC,...",1680,Bags,405.0,680400
1,TMT steel – Kamdhenu / SAIL Fe500D (incl. 3% w...,8250,Kg,64.0,528000
2,Binding wire – 18 gauge,100,Kg,82.0,8200
3,"River sand – for concrete, masonry and plaster",85,Cum,2850.0,242250
4,"Aggregate – 20 mm and 12 mm graded, crushed stone",92,Cum,1850.0,170200
5,"Bricks – first-class red clay, 230x110x75 mm",37000,Nos.,9.0,333000
6,Vitrified tiles – 600x600 mm double-charge (Ka...,2050,Sq.ft.,72.0,147600
7,Ceramic wall tiles – 300x450 mm (toilets and k...,760,Sq.ft.,46.0,34960
8,"Tile adhesive, grout and spacers",1,LS,38000.0,38000
9,"Paints – putty, primer, interior emulsion and ...",8700,Sq.ft.,11.5,100050



CONTRACTOR: Gomes Dias Builders Pvt. Ltd.
PROJECT: Proposed G+1 Residential House for Mr. Siddharth V. Kamat
TOTAL AMOUNT: 5646300
TIMELINE: 11 Months from the date of commencement
WARRANTY: Defects liability period of 24 months from the date of handing over for workmanship defects; 5 years for terrace waterproofing leakage.
PAYMENT: Advance Payment: 20% on acceptance of quotation / issue of work order; During Execution: 70% released stage-wise; On Completion: 10% on completion and handing over of the building

ITEMS:


,description,quantity,unit,rate,amount
0,Site clearance and preparation – vegetation an...,1,LS,None,None
1,"Earthwork excavation – footings, plinth beams ...",140,Cum,None,None
2,Plain cement concrete – M10 bed under footings...,14,Cum,None,None
3,RCC work – M25 grade (UltraTech OPC 53) for fo...,85,Cum,None,None
4,Reinforcement steel work – Fe500D TMT bars (JS...,8000,Kg,None,None
5,Brick masonry – 230 mm external and 115 mm int...,72,Cum,None,None
6,Internal plastering – 12 mm cement mortar 1:4 ...,5800,Sq.ft.,None,None
7,Flooring work – 600x600 mm double-charge vitri...,1950,Sq.ft.,None,None
8,"Painting work – internal: putty, primer and 2 ...",8700,Sq.ft.,None,None
9,Plumbing and electrical work – concealed plumb...,1,LS,None,None



CONTRACTOR: Konkan Crest Constructions
PROJECT: Proposed G+1 Residential House for Mr. Siddharth V. Kamat
TOTAL AMOUNT: 5469241
TIMELINE: 10 Months from the date of commencement (excluding monsoon stoppage of 15 days, if any)
WARRANTY: Defects liability period of 12 months from the date of handing over; the contractor shall rectify workmanship defects at no extra cost during this period.
PAYMENT: Advance Payment: 15% on acceptance of quotation / issue of work order; During Execution: 75% – released stage-wise (10% - completion of foundation and plinth beam, 15% - completion of ground floor slab, 15% - completion of first floor slab, 15% - completion of roof slab, 10% - completion of brickwork and plastering, 10% - completion of flooring, tiling, plumbing and electrical); On Completion: 10% on completion and handing over of the building

ITEMS:


,description,quantity,unit,rate,amount
0,Site clearance and preparation – removal of ve...,1,LS,45000,45000
1,"Earthwork excavation – footings, plinth beams,...",140,Cum,380,53200
2,"Plain cement concrete – M10 (1:3:6), 100 mm th...",14,Cum,6800,95200
3,"RCC work – M25 grade (ACC OPC 53, 20 mm graded...",85,Cum,12850,1092250
4,Reinforcement steel work – Fe500D TMT bars (Ta...,8000,Kg,84,672000
5,Brick masonry – 230 mm external and 115 mm int...,72,Cum,8650,622800
6,Internal plastering – 12 mm thick cement morta...,5800,Sq.ft.,37,214600
7,Flooring work – 600x600 mm double-charge vitri...,1950,Sq.ft.,148,288600
8,"Painting work – internal: 2 coats putty, 1 coa...",8700,Sq.ft.,25,217500
9,Plumbing and electrical work – concealed CPVC ...,1,LS,410000,410000


In [11]:

all_items = []

for filename, data in extracted_quotations.items():
    contractor = data.get("contractor")

    for item in data.get("items", []):
        all_items.append({
            "contractor": contractor,
            "description": item.get("description"),
            "quantity": item.get("quantity"),
            "unit": item.get("unit"),
            "rate": item.get("rate"),
            "amount": item.get("amount")
        })

items_df = pd.DataFrame(all_items)

display(items_df)


,contractor,description,quantity,unit,rate,amount
0,Vaz Patil Constructions LLP,"Cement – Ambuja OPC 53 Grade, 50 kg bags (RCC,...",1680,Bags,405.0,680400.0
1,Vaz Patil Constructions LLP,TMT steel – Kamdhenu / SAIL Fe500D (incl. 3% w...,8250,Kg,64.0,528000.0
2,Vaz Patil Constructions LLP,Binding wire – 18 gauge,100,Kg,82.0,8200.0
3,Vaz Patil Constructions LLP,"River sand – for concrete, masonry and plaster",85,Cum,2850.0,242250.0
4,Vaz Patil Constructions LLP,"Aggregate – 20 mm and 12 mm graded, crushed stone",92,Cum,1850.0,170200.0
...,...,...,...,...,...,...
57,Konkan Crest Constructions,External plastering – 20 mm thick cement morta...,2800,Sq.ft.,42.0,117600.0
58,Konkan Crest Constructions,Waterproofing work – terrace (brickbat coba wi...,1300,Sq.ft.,68.0,88400.0
59,Konkan Crest Constructions,"Doors, windows, grills and railings – 10 Nos. ...",1,LS,320000.0,320000.0
60,Konkan Crest Constructions,"Septic tank (3,000 L, two-chamber brick masonr...",1,LS,135000.0,135000.0


In [12]:

normalization_prompt = f"""
You are a construction material and quotation analyst.

Analyze these contractor quotation items:

{items_df.to_json(orient="records")}

For every item, return:
- contractor
- description (original description)
- standardized_item (standard construction item name)
- category (e.g. Cement, Steel, Labour, Electrical, Plumbing)
- comparable (true or false)
- reason (brief explanation)

Standardize descriptions only when items are technically comparable.
Do not treat different grades, specifications or units as equivalent.

Return valid JSON only as a list of objects.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=normalization_prompt
)

normalized_items = json.loads(response.text.strip().replace("```json", "").replace("```", "").strip())

normalized_df = pd.DataFrame(normalized_items)

display(normalized_df)


,contractor,description,standardized_item,category,comparable,reason
0,Vaz Patil Constructions LLP,"Cement – Ambuja OPC 53 Grade, 50 kg bags (RCC,...",Cement Ambuja OPC 53 Grade (50 kg bag),Cement,True,"Standard specification cement bag item, direct..."
1,Vaz Patil Constructions LLP,TMT steel – Kamdhenu / SAIL Fe500D (incl. 3% w...,TMT Steel Fe500D,Steel,True,Standard TMT steel material item by weight.
2,Vaz Patil Constructions LLP,Binding wire – 18 gauge,Binding Wire 18 gauge,Steel,False,Material-only item not separately quoted by ot...
3,Vaz Patil Constructions LLP,"River sand – for concrete, masonry and plaster",River Sand,Aggregates,False,Material-only item not separately quoted by ot...
4,Vaz Patil Constructions LLP,"Aggregate – 20 mm and 12 mm graded, crushed stone",Graded Crushed Stone Aggregate (20mm & 12mm),Aggregates,False,Material-only item not separately quoted by ot...
...,...,...,...,...,...,...
57,Konkan Crest Constructions,External plastering – 20 mm thick cement morta...,External Plastering Work,Plastering,True,Standard external plastering item with matchin...
58,Konkan Crest Constructions,Waterproofing work – terrace (brickbat coba wi...,Waterproofing Application Work,Waterproofing,True,Standard waterproofing application item with m...
59,Konkan Crest Constructions,"Doors, windows, grills and railings – 10 Nos. ...","Doors, Windows, Grills and Railings",Carpentry & Fabrication,False,"Combined supply and fixing item, whereas Vaz P..."
60,Konkan Crest Constructions,"Septic tank (3,000 L, two-chamber brick masonr...","Septic Tank, Soak Pit and Sump Construction",Civil Works,False,"Combined material and construction item, where..."


In [13]:

price_df = pd.DataFrame([
    {
        "Contractor": data["contractor"],
        "Total Price": data["total_amount"]
    }
    for data in extracted_quotations.values()
])

price_df["Total Price"] = pd.to_numeric(
    price_df["Total Price"], errors="coerce"
)

display(price_df)


,Contractor,Total Price
0,Vaz Patil Constructions LLP,5172318
1,Gomes Dias Builders Pvt. Ltd.,5646300
2,Konkan Crest Constructions,5469241


In [14]:

lowest_price = price_df["Total Price"].min()

price_score_df = price_df.copy()

price_score_df["Price Score"] = (
    lowest_price / price_score_df["Total Price"]
) * 100

display(price_score_df.round(2))


,Contractor,Total Price,Price Score
0,Vaz Patil Constructions LLP,5172318,100.00
1,Gomes Dias Builders Pvt. Ltd.,5646300,91.61
2,Konkan Crest Constructions,5469241,94.57


In [15]:
scope_prompt = f"""
You are an expert construction quotation analyst.

Evaluate the scope coverage of each contractor using these
extracted quotation items:

{items_df.to_json(orient="records")}

For each contractor, assign a Scope Score out of 100 based on:
- Coverage of construction work items
- Completeness of the quoted work
- Missing or excluded work

Return valid JSON only in this format:
[
    {{"Contractor": "Name", "Scope Score": 95}}
]

Do not invent quotation details.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=scope_prompt
)

scope_scores = pd.DataFrame(
    json.loads(
        response.text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
)

display(scope_scores)

,Contractor,Scope Score
0,Vaz Patil Constructions LLP,92
1,Gomes Dias Builders Pvt. Ltd.,55
2,Konkan Crest Constructions,90


In [16]:
technical_prompt = f"""
You are a construction materials expert.

Evaluate the technical specification quality of each contractor's
quoted materials using the following normalized items:

{normalized_df.to_json(orient="records")}

Consider:
- Material grade and specifications
- Quality of the specified materials
- Whether specifications are clearly mentioned

Give each contractor a Technical Specification Score out of 100.

Return valid JSON only:
[
    {{"Contractor": "Name", "Technical Specification Score": 90}}
]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=technical_prompt
)

technical_scores = pd.DataFrame(
    json.loads(
        response.text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
)

display(technical_scores)

,Contractor,Technical Specification Score
0,Vaz Patil Constructions LLP,85
1,Gomes Dias Builders Pvt. Ltd.,60
2,Konkan Crest Constructions,92


In [17]:
brand_prompt = f"""
You are a construction materials analyst.

Evaluate each contractor's quotation based on the identification
of material brands.

Use these normalized quotation items:
{normalized_df.to_json(orient="records")}

Consider:
- Whether recognized brands are specified
- Whether brand names are clearly identified
- Whether material brands are traceable

Give each contractor a Brand Identification Score out of 100.

Return valid JSON only:
[
    {{"Contractor": "Name", "Brand Identification Score": 90}}
]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=brand_prompt
)

brand_scores = pd.DataFrame(
    json.loads(
        response.text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
)

display(brand_scores)

,Contractor,Brand Identification Score
0,Vaz Patil Constructions LLP,85
1,Gomes Dias Builders Pvt. Ltd.,80
2,Konkan Crest Constructions,90


In [18]:
completeness_prompt = f"""
You are a construction quotation analyst.

Evaluate the information completeness of each contractor's quotation.

Data:
{json.dumps(extracted_quotations, indent=2)}

Consider:
- Contractor details
- Project details
- Item descriptions and quantities
- Rates and amounts
- GST details
- Completion timeline
- Warranty details
- Payment terms

Give each contractor a Completeness Score out of 100.

Return valid JSON only:
[
    {{"Contractor": "Name", "Completeness (%)": 85}}
]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=completeness_prompt
)

completeness_df = pd.DataFrame(
    json.loads(
        response.text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
)

display(completeness_df)

,Contractor,Completeness (%)
0,Vaz Patil Constructions LLP,95
1,Gomes Dias Builders Pvt. Ltd.,75
2,Konkan Crest Constructions,98


In [19]:
timeline_df = pd.DataFrame([
    {
        "Contractor": data["contractor"],
        "Completion Period": data["completion_period"]
    }
    for data in extracted_quotations.values()
])

display(timeline_df)

,Contractor,Completion Period
0,Vaz Patil Constructions LLP,9 Months from the date of commencement
1,Gomes Dias Builders Pvt. Ltd.,11 Months from the date of commencement
2,Konkan Crest Constructions,10 Months from the date of commencement (exclu...


In [20]:
timeline_df = pd.DataFrame([
    {
        "Contractor": data["contractor"],
        "Completion Period": data["completion_period"]
    }
    for data in extracted_quotations.values()
])

display(timeline_df)

,Contractor,Completion Period
0,Vaz Patil Constructions LLP,9 Months from the date of commencement
1,Gomes Dias Builders Pvt. Ltd.,11 Months from the date of commencement
2,Konkan Crest Constructions,10 Months from the date of commencement (exclu...


In [21]:
timeline_prompt = f"""
You are a construction project management expert.

Evaluate these contractor completion timelines:

{timeline_df.to_json(orient="records")}

Score each contractor out of 100.
A shorter realistic completion period should receive a
higher score. Consider whether the timeline is clearly stated.

Return valid JSON only:
[
    {{"Contractor": "Name", "Timeline Score": 90}}
]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=timeline_prompt
)

timeline_score_df = pd.DataFrame(
    json.loads(
        response.text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
)

display(timeline_score_df)

,Contractor,Timeline Score
0,Vaz Patil Constructions LLP,95
1,Konkan Crest Constructions,85
2,Gomes Dias Builders Pvt. Ltd.,80


In [22]:
warranty_df = pd.DataFrame([
    {
        "Contractor": data["contractor"],
        "Warranty": data["warranty"]
    }
    for data in extracted_quotations.values()
])

display(warranty_df)

,Contractor,Warranty
0,Vaz Patil Constructions LLP,Defects liability period of 12 months from the...
1,Gomes Dias Builders Pvt. Ltd.,Defects liability period of 24 months from the...
2,Konkan Crest Constructions,Defects liability period of 12 months from the...


In [23]:
warranty_months = {
    "VAZ PATIL CONSTRUCTIONS LLP": 12,
    "Gomes Dias Builders Pvt. Ltd.": 24,
    "Konkan Crest Constructions": 12
}

warranty_score_df = pd.DataFrame([
    {
        "Contractor": contractor,
        "Warranty Score": months / 24 * 100
    }
    for contractor, months in warranty_months.items()
])

display(warranty_score_df)

,Contractor,Warranty Score
0,VAZ PATIL CONSTRUCTIONS LLP,50.0
1,Gomes Dias Builders Pvt. Ltd.,100.0
2,Konkan Crest Constructions,50.0


In [24]:
payment_df = pd.DataFrame([
    {
        "Contractor": data["contractor"],
        "Payment Terms": data["payment_terms"]
    }
    for data in extracted_quotations.values()
])

display(payment_df)

,Contractor,Payment Terms
0,Vaz Patil Constructions LLP,Advance Payment: 25% on acceptance of quotatio...
1,Gomes Dias Builders Pvt. Ltd.,Advance Payment: 20% on acceptance of quotatio...
2,Konkan Crest Constructions,Advance Payment: 15% on acceptance of quotatio...


In [25]:
payment_prompt = f"""
You are a construction contract analyst.

Evaluate the payment terms offered by each contractor:

{payment_df.to_json(orient="records")}

Consider:
- Advance payment percentage
- Stage-wise payment structure
- Payment flexibility
- Clarity and transparency

Give each contractor a Payment Score out of 100.

Return valid JSON only:
[
    {{"Contractor": "Name", "Payment Score": 90}}
]
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=payment_prompt
)

payment_score_df = pd.DataFrame(
    json.loads(
        response.text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
)

display(payment_score_df)

,Contractor,Payment Score
0,Vaz Patil Constructions LLP,75
1,Gomes Dias Builders Pvt. Ltd.,80
2,Konkan Crest Constructions,95


In [31]:
# 1. Prepare material scores
material_score_df = technical_scores.merge(
    brand_scores,
    on="Contractor"
)

material_score_df["Material Score"] = (
    material_score_df["Technical Specification Score"] * 0.50
    + material_score_df["Brand Identification Score"] * 0.25
) / 0.75

# 2. Merge all seven criteria
final_df = price_score_df[
    ["Contractor", "Price Score"]
].merge(
    scope_scores[["Contractor", "Scope Score"]],
    on="Contractor"
).merge(
    material_score_df[["Contractor", "Material Score"]],
    on="Contractor"
).merge(
    completeness_df[["Contractor", "Completeness (%)"]],
    on="Contractor"
).merge(
    timeline_score_df[["Contractor", "Timeline Score"]],
    on="Contractor"
).merge(
    warranty_score_df,
    on="Contractor"
).merge(
    payment_score_df[["Contractor", "Payment Score"]],
    on="Contractor"
)

# 3. Apply finalized weights
final_df["Overall Score"] = (
    final_df["Price Score"] * 0.25
    + final_df["Scope Score"] * 0.25
    + final_df["Material Score"] * 0.15
    + final_df["Completeness (%)"] * 0.10
    + final_df["Timeline Score"] * 0.10
    + final_df["Warranty Score"] * 0.10
    + final_df["Payment Score"] * 0.05
)

# 4. Rank contractors
final_df = final_df.sort_values(
    by="Overall Score",
    ascending=False
).reset_index(drop=True)

final_df.insert(0, "Rank", range(1, len(final_df) + 1))

display(final_df.round(2))

,Rank,Contractor,Price Score,Scope Score,Material Score,Completeness (%),Timeline Score,Warranty Score,Payment Score,Overall Score
0,1,Konkan Crest Constructions,94.57,90,91.33,98,85,50.0,95,87.89
1,2,Gomes Dias Builders Pvt. Ltd.,91.61,55,66.67,75,80,100.0,80,76.15


In [35]:
# 1. Prepare material scores
material_score_df = technical_scores.merge(
    brand_scores,
    on="Contractor"
)

material_score_df["Material Score"] = (
    material_score_df["Technical Specification Score"] * 0.50
    + material_score_df["Brand Identification Score"] * 0.25
) / 0.75

# 2. Merge all seven criteria
final_df = price_score_df[
    ["Contractor", "Price Score"]
].merge(
    scope_scores[["Contractor", "Scope Score"]],
    on="Contractor"
).merge(
    material_score_df[["Contractor", "Material Score"]],
    on="Contractor"
).merge(
    completeness_df[["Contractor", "Completeness (%)"]],
    on="Contractor"
).merge(
    timeline_score_df[["Contractor", "Timeline Score"]],
    on="Contractor"
).merge(
    warranty_score_df,
    on="Contractor"
).merge(
    payment_score_df[["Contractor", "Payment Score"]],
    on="Contractor"
)

# 3. Apply finalized weights
final_df["Overall Score"] = (
    final_df["Price Score"] * 0.25
    + final_df["Scope Score"] * 0.25
    + final_df["Material Score"] * 0.15
    + final_df["Completeness (%)"] * 0.10
    + final_df["Timeline Score"] * 0.10
    + final_df["Warranty Score"] * 0.10
    + final_df["Payment Score"] * 0.05
)

# 4. Rank contractors
final_df = final_df.sort_values(
    by="Overall Score",
    ascending=False
).reset_index(drop=True)

final_df.insert(0, "Rank", range(1, len(final_df) + 1))

display(final_df.round(2))

,Rank,Contractor,Price Score,Scope Score,Material Score,Completeness (%),Timeline Score,Warranty Score,Payment Score,Overall Score
0,1,Konkan Crest Constructions,94.57,90,91.33,98,85,50.0,95,87.89
1,2,Gomes Dias Builders Pvt. Ltd.,91.61,55,66.67,75,80,100.0,80,76.15


In [29]:
dataframes = {
    "price_score_df": price_score_df,
    "scope_scores": scope_scores,
    "technical_scores": technical_scores,
    "brand_scores": brand_scores,
    "completeness_df": completeness_df,
    "timeline_score_df": timeline_score_df,
    "warranty_score_df": warranty_score_df,
    "payment_score_df": payment_score_df
}

for name, df in dataframes.items():
    print(f"\n{name}")
    print(df["Contractor"].tolist())


price_score_df
['Vaz Patil Constructions LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']

scope_scores
['Vaz Patil Constructions LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']

technical_scores
['Vaz Patil Constructions LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']

brand_scores
['Vaz Patil Constructions LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']

completeness_df
['Vaz Patil Constructions LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']

timeline_score_df
['Vaz Patil Constructions LLP', 'Konkan Crest Constructions', 'Gomes Dias Builders Pvt. Ltd.']

warranty_score_df
['VAZ PATIL CONSTRUCTIONS LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']

payment_score_df
['Vaz Patil Constructions LLP', 'Gomes Dias Builders Pvt. Ltd.', 'Konkan Crest Constructions']


In [30]:
warranty_score_df["Contractor"] = (
    warranty_score_df["Contractor"].str.title()
)

display(warranty_score_df)

,Contractor,Warranty Score
0,Vaz Patil Constructions Llp,50.0
1,Gomes Dias Builders Pvt. Ltd.,100.0
2,Konkan Crest Constructions,50.0


In [34]:
warranty_score_df["Contractor"] = (
    warranty_score_df["Contractor"].str.title()
)

display(warranty_score_df)

,Contractor,Warranty Score
0,Vaz Patil Constructions Llp,50.0
1,Gomes Dias Builders Pvt. Ltd.,100.0
2,Konkan Crest Constructions,50.0


In [36]:
dataframes = {
    "price_score_df": price_score_df,
    "scope_scores": scope_scores,
    "technical_scores": technical_scores,
    "brand_scores": brand_scores,
    "completeness_df": completeness_df,
    "timeline_score_df": timeline_score_df,
    "warranty_score_df": warranty_score_df,
    "payment_score_df": payment_score_df
}

for df in dataframes.values():
    df["Contractor"] = (
        df["Contractor"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

for name, df in dataframes.items():
    print(name, df["Contractor"].tolist())

price_score_df ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']
scope_scores ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']
technical_scores ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']
brand_scores ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']
completeness_df ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']
timeline_score_df ['VAZ PATIL CONSTRUCTIONS LLP', 'KONKAN CREST CONSTRUCTIONS', 'GOMES DIAS BUILDERS PVT. LTD.']
warranty_score_df ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']
payment_score_df ['VAZ PATIL CONSTRUCTIONS LLP', 'GOMES DIAS BUILDERS PVT. LTD.', 'KONKAN CREST CONSTRUCTIONS']


In [37]:
# Recalculate material scores
material_score_df = technical_scores.merge(
    brand_scores, on="Contractor"
)

material_score_df["Material Score"] = (
    material_score_df["Technical Specification Score"] * 0.50
    + material_score_df["Brand Identification Score"] * 0.25
) / 0.75

# Start with price scores
final_df = price_score_df[
    ["Contractor", "Price Score"]
].copy()

# Merge remaining scores
score_tables = [
    (scope_scores, ["Scope Score"]),
    (material_score_df, ["Material Score"]),
    (completeness_df, ["Completeness (%)"]),
    (timeline_score_df, ["Timeline Score"]),
    (warranty_score_df, ["Warranty Score"]),
    (payment_score_df, ["Payment Score"])
]

for df, columns in score_tables:
    final_df = final_df.merge(
        df[["Contractor"] + columns],
        on="Contractor",
        how="inner"
    )
    print("Contractors after merge:", len(final_df))

# Calculate overall score
final_df["Overall Score"] = (
    final_df["Price Score"] * 0.25
    + final_df["Scope Score"] * 0.25
    + final_df["Material Score"] * 0.15
    + final_df["Completeness (%)"] * 0.10
    + final_df["Timeline Score"] * 0.10
    + final_df["Warranty Score"] * 0.10
    + final_df["Payment Score"] * 0.05
)

# Rank contractors
final_df = final_df.sort_values(
    "Overall Score", ascending=False
).reset_index(drop=True)

final_df.insert(0, "Rank", range(1, len(final_df) + 1))

display(final_df.round(2))

Contractors after merge: 3
Contractors after merge: 3
Contractors after merge: 3
Contractors after merge: 3
Contractors after merge: 3
Contractors after merge: 3


,Rank,Contractor,Price Score,Scope Score,Material Score,Completeness (%),Timeline Score,Warranty Score,Payment Score,Overall Score
0,1,VAZ PATIL CONSTRUCTIONS LLP,100.00,92,85.00,95,95,50.0,75,88.50
1,2,KONKAN CREST CONSTRUCTIONS,94.57,90,91.33,98,85,50.0,95,87.89
2,3,GOMES DIAS BUILDERS PVT. LTD.,91.61,55,66.67,75,80,100.0,80,76.15
